# 01 - Prophet Model

Ce notebook entraine un modele Prophet pour la prevision des ventes.

**Outputs:**
- `models/artifacts/prophet_model_YYYYMMDD.pkl`
- `models/artifacts/prophet_config_YYYYMMDD.json`
- `models/metrics/prophet_metrics_YYYYMMDD.json`

In [ ]:
import sys
import json
import datetime
import warnings
from pathlib import Path
from math import sqrt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# Setup project path - SAME AS train.py
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
HORIZON = 14
DATE_COL = 'date'
TARGET_COL = 'value'
TODAY = datetime.datetime.now().strftime("%Y%m%d")

# Paths
ARTIFACTS_PATH = PROJECT_ROOT / "models" / "artifacts"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics"
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# Import project config
try:
    from src.config import TRAIN_CSV, GROUP_COLS
except ImportError:
    TRAIN_CSV = PROJECT_ROOT / "data" / "interim" / "train.csv"
    GROUP_COLS = ['store_id', 'product_id']

try:
    from src.data.clean import clean_dataframe
    CLEAN_AVAILABLE = True
except ImportError:
    CLEAN_AVAILABLE = False

print(f"HORIZON={HORIZON}, TODAY={TODAY}")

In [ ]:
# Find data file - SAME LOGIC AS train.py
def find_data_file():
    candidates = [
        TRAIN_CSV,
        PROJECT_ROOT / "data" / "processed" / "uploaded_generated_training_10950_features.csv",
        PROJECT_ROOT / "data" / "processed" / "train_features.csv",
    ]
    for folder in ["interim", "processed", "raw"]:
        folder_path = PROJECT_ROOT / "data" / folder
        if folder_path.exists():
            for f in folder_path.glob("*.csv"):
                candidates.append(f)
    
    for p in candidates:
        if p and p.exists():
            return p
    raise FileNotFoundError("No data file found")

DATA_PATH = find_data_file()
print(f"Data: {DATA_PATH}")

## 1. Chargement et Nettoyage

In [ ]:
# Load data
df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
print(f"Shape: {df.shape}")

# Clean - SAME AS train.py
if CLEAN_AVAILABLE:
    df = clean_dataframe(df, date_col=DATE_COL, value_col=TARGET_COL, 
                         fill_strategy='ffill', outlier_threshold=None)

# Remap columns
if 'store_id' not in df.columns and 'id' in df.columns:
    df = df.rename(columns={'id': 'store_id'})

# Ensure numeric target
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce')
df = df.dropna(subset=[TARGET_COL])

# Winsorization
lower = df[TARGET_COL].quantile(0.005)
upper = df[TARGET_COL].quantile(0.995)
df[TARGET_COL] = df[TARGET_COL].clip(lower=max(0, lower), upper=upper)

print(f"Clean shape: {df.shape}")
print(f"Target: mean={df[TARGET_COL].mean():.2f}, std={df[TARGET_COL].std():.2f}")

In [ ]:
# Aggregate by date for Prophet
df_agg = df.groupby(DATE_COL)[TARGET_COL].sum().reset_index()
df_agg.columns = ['ds', 'y']
df_agg = df_agg.sort_values('ds').reset_index(drop=True)

# Ensure positive
if df_agg['y'].min() <= 0:
    df_agg['y'] = df_agg['y'] - df_agg['y'].min() + 1

print(f"Aggregated: {len(df_agg)} daily records")
print(f"Date range: {df_agg['ds'].min()} to {df_agg['ds'].max()}")

## 2. Train/Test Split

In [ ]:
# Temporal split - SAME AS train.py
n_test = max(HORIZON, int(len(df_agg) * 0.15))
cutoff_idx = len(df_agg) - n_test

df_train = df_agg.iloc[:cutoff_idx].copy()
df_test = df_agg.iloc[cutoff_idx:].copy()

print(f"Train: {len(df_train)} | Test: {len(df_test)}")

## 3. Prophet Training

In [ ]:
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# Determine seasonality
cv = df_train['y'].std() / (df_train['y'].mean() + 1e-8)
seasonality_mode = 'multiplicative' if cv > 0.5 else 'additive'
n_days = len(df_train)

print(f"CV: {cv:.3f} -> {seasonality_mode} seasonality")
print(f"Data spans {n_days} days")

In [ ]:
# Train Prophet
model = Prophet(
    seasonality_mode=seasonality_mode,
    yearly_seasonality=(n_days >= 365),
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=0.1,
)

if n_days >= 60:
    model.add_seasonality(name='monthly', period=30.5, fourier_order=5)

print("Training Prophet...")
model.fit(df_train)
print("Done!")

## 4. Predictions & Metrics

In [ ]:
# Metrics function - SAME AS train.py
def compute_metrics(y_true, y_pred):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    bias = float(np.mean(y_pred - y_true))
    
    # Safe MAPE
    mask = np.abs(y_true) > 1.0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.sum() > 0 else None
    
    # sMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom_safe = np.where(denom > 0, denom, 1.0)
    smape = float(np.mean(np.abs(y_true - y_pred) / denom_safe) * 100)
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Bias': bias, 'MAPE': mape, 'sMAPE': smape}

In [ ]:
# Predict on train
forecast_train = model.predict(df_train[['ds']])
y_train_true = df_train['y'].values
y_train_pred = forecast_train['yhat'].clip(lower=0).values
train_metrics = compute_metrics(y_train_true, y_train_pred)

# Predict on test
forecast_test = model.predict(df_test[['ds']])
y_test_true = df_test['y'].values
y_test_pred = forecast_test['yhat'].clip(lower=0).values
test_metrics = compute_metrics(y_test_true, y_test_pred)

# Bias-corrected
y_test_corrected = y_test_pred - test_metrics['Bias']
test_metrics_corrected = compute_metrics(y_test_true, y_test_corrected)

In [ ]:
# Cross-validation
try:
    horizon_cv = f"{min(HORIZON, n_days // 4)} days"
    initial_cv = f"{max(n_days // 2, 30)} days"
    period_cv = f"{max(HORIZON // 2, 7)} days"
    
    df_cv = cross_validation(model, initial=initial_cv, period=period_cv, 
                              horizon=horizon_cv, parallel="threads")
    cv_perf = performance_metrics(df_cv)
    cv_mae = cv_perf['mae'].mean()
    cv_mae_std = cv_perf['mae'].std()
    cv_rmse = cv_perf['rmse'].mean()
    print(f"CV MAE: {cv_mae:.4f} (+/- {cv_mae_std:.4f})")
except Exception as e:
    print(f"CV failed: {e}")
    cv_mae, cv_mae_std, cv_rmse = None, None, None

In [ ]:
# Overfit ratio
overfit_ratio = test_metrics['RMSE'] / train_metrics['RMSE'] if train_metrics['RMSE'] > 0 else float('inf')

print("\n" + "="*60)
print("[Prophet] RESULTS")
print("="*60)
print(f"  Train  -> MAE={train_metrics['MAE']:.4f}  RMSE={train_metrics['RMSE']:.4f}  R2={train_metrics['R2']:.4f}")
print(f"  Test   -> MAE={test_metrics['MAE']:.4f}  RMSE={test_metrics['RMSE']:.4f}  R2={test_metrics['R2']:.4f}")
print(f"  Test*  -> MAE={test_metrics_corrected['MAE']:.4f}  R2={test_metrics_corrected['R2']:.4f}  (* bias-corrected)")
print(f"  Bias   -> train={train_metrics['Bias']:.4f}  test={test_metrics['Bias']:.4f}")
print(f"  MAPE   -> {test_metrics['MAPE']:.2f}%" if test_metrics['MAPE'] else "  MAPE   -> N/A")
print(f"  sMAPE  -> {test_metrics['sMAPE']:.2f}%")
print(f"  Overfit ratio: {overfit_ratio:.2f}")
if cv_mae:
    print(f"  CV     -> MAE={cv_mae:.4f} (+/- {cv_mae_std:.4f})")
print("="*60)

## 5. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Forecast
ax1 = axes[0, 0]
ax1.plot(df_train['ds'], df_train['y'], label='Train', alpha=0.7)
ax1.plot(df_test['ds'], df_test['y'], label='Actual', linewidth=2)
ax1.plot(df_test['ds'], y_test_pred, label='Predicted', linestyle='--')
ax1.fill_between(df_test['ds'], forecast_test['yhat_lower'], forecast_test['yhat_upper'], alpha=0.2)
ax1.set_title('Prophet Forecast')
ax1.legend()

# 2. Scatter
ax2 = axes[0, 1]
ax2.scatter(y_test_true, y_test_pred, alpha=0.6)
max_val = max(y_test_true.max(), y_test_pred.max())
ax2.plot([0, max_val], [0, max_val], 'r--', label='Perfect')
ax2.set_xlabel('Actual')
ax2.set_ylabel('Predicted')
ax2.set_title(f'Scatter (R2={test_metrics["R2"]:.3f})')
ax2.legend()

# 3. Residuals
ax3 = axes[1, 0]
residuals = y_test_true - y_test_pred
ax3.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
ax3.axvline(x=0, color='red', linestyle='--')
ax3.set_title(f'Residuals (Bias={test_metrics["Bias"]:.2f})')

# 4. Components
ax4 = axes[1, 1]
ax4.text(0.5, 0.5, f"Model: Prophet\nSeasonality: {seasonality_mode}\nMAE: {test_metrics['MAE']:.2f}\nR2: {test_metrics['R2']:.3f}", 
         ha='center', va='center', fontsize=14, transform=ax4.transAxes)
ax4.axis('off')

plt.tight_layout()
plt.savefig(METRICS_PATH / f"prophet_plots_{TODAY}.png", dpi=150)
plt.show()

## 6. Save Artifacts & Metrics

In [ ]:
# Compile all metrics
all_metrics = {
    'model_type': 'Prophet',
    'timestamp': datetime.datetime.now().isoformat(),
    'horizon': HORIZON,
    'seasonality_mode': seasonality_mode,
    'num_train_samples': len(df_train),
    'num_test_samples': len(df_test),
    
    # Train
    'MAE_train': float(train_metrics['MAE']),
    'RMSE_train': float(train_metrics['RMSE']),
    'R2_train': float(train_metrics['R2']),
    'Bias_train': float(train_metrics['Bias']),
    
    # Test
    'MAE_test': float(test_metrics['MAE']),
    'RMSE_test': float(test_metrics['RMSE']),
    'R2_test': float(test_metrics['R2']),
    'Bias_test': float(test_metrics['Bias']),
    'MAPE_test': test_metrics['MAPE'],
    'sMAPE_test': float(test_metrics['sMAPE']),
    
    # Corrected
    'MAE_test_corrected': float(test_metrics_corrected['MAE']),
    'R2_test_corrected': float(test_metrics_corrected['R2']),
    
    # CV
    'CV_mae_mean': cv_mae,
    'CV_mae_std': cv_mae_std,
    'CV_rmse_mean': cv_rmse,
    
    # Overfit
    'overfit_ratio': float(overfit_ratio),
}

# Save metrics JSON
metrics_file = METRICS_PATH / f"prophet_metrics_{TODAY}.json"
with open(metrics_file, 'w') as f:
    json.dump(all_metrics, f, indent=2, default=str)
print(f"Metrics saved: {metrics_file}")

# Save model
model_file = ARTIFACTS_PATH / f"prophet_model_{TODAY}.pkl"
joblib.dump(model, model_file)
print(f"Model saved: {model_file}")

# Save config
config = {
    'horizon': HORIZON,
    'seasonality_mode': seasonality_mode,
    'yearly_seasonality': n_days >= 365,
    'weekly_seasonality': True,
    'changepoint_prior_scale': 0.05,
    'bias_correction': -test_metrics['Bias'],
    'data_path': str(DATA_PATH),
}
config_file = ARTIFACTS_PATH / f"prophet_config_{TODAY}.json"
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Config saved: {config_file}")

## 7. Final Summary

In [ ]:
print("\n" + "="*60)
print("PROPHET MODEL - FINAL SUMMARY")
print("="*60)
print(f"  Model        : Prophet ({seasonality_mode})")
print(f"  Horizon      : {HORIZON} days")
print(f"  Train samples: {len(df_train)}")
print(f"  Test samples : {len(df_test)}")
print()
print("  Test Metrics:")
print(f"    MAE   : {test_metrics['MAE']:.4f}")
print(f"    RMSE  : {test_metrics['RMSE']:.4f}")
print(f"    R2    : {test_metrics['R2']:.4f}")
print(f"    MAPE  : {test_metrics['MAPE']:.2f}%" if test_metrics['MAPE'] else "    MAPE  : N/A")
print(f"    sMAPE : {test_metrics['sMAPE']:.2f}%")
print()
print("  Artifacts:")
print(f"    {model_file}")
print(f"    {config_file}")
print(f"    {metrics_file}")
print("="*60)